# Chapter `2.2.1` - Runtime Context Management

In [1]:
from os import getenv
from dotenv import load_dotenv
from IPython.display import Markdown

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime

import warnings

...

Ellipsis

### Setup and configuration

In [2]:
load_dotenv()

try:
    OLLAMA_MODEL = getenv("OLLAMA_MODEL", "")
    if not len(OLLAMA_MODEL):
        raise EnvironmentError("Missing Ollama model configuration in environment.")
except EnvironmentError as ee:
    print(f"ERROR: {ee}")

In [3]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph / Pydantic noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")

load_dotenv()

# TODO: Environment variable config.

True

### Context definition

In [4]:
@dataclass
class ColourContext:
    favourite_colour: str = "black"
    least_favourite_colour: str = "pink"

### Agent invocation

In [5]:
agent = create_agent(
    model=OLLAMA_MODEL,
    context_schema=ColourContext  
)

In [6]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [7]:
Markdown(response["messages"][-1].content)

I don't know what your favourite colour is because we haven't discussed it yet! 

Since I don't have access to your personal life or memories, you'll have to tell me. **What is it?**

> #### Even though we have provided the `context_schema`, the agent is **unable** to access the runtime context.

### Accessing runtime context
- In order to ensure that the agent is able to access the **runtime context**, we need to provide the proper **tools**.

In [8]:
@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour  # type: ignore


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour  # type: ignore

In [9]:
agent2 = create_agent(
    model=OLLAMA_MODEL,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [10]:
response = agent2.invoke(
    {"messages": [HumanMessage(content="What are favourite and least favourite colours?")]},
    context=ColourContext()
)

Markdown(response["messages"][-1].content)

Your favourite colour is black, and your least favourite colour is pink.

In [12]:
response = agent2.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="cyan")
)

Markdown(response["messages"][-1].content)

Your favourite colour is cyan.